In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

df = pd.read_csv("../modeling/datasets/kkbox_train.csv")
df.shape


(480000, 21)

## Shape

In [2]:
print(f"rows={df.shape[0]}, columns={df.shape[1]}")
print(f"memory_mb={df.memory_usage(deep=True).sum() / 1e6:.2f}")


rows=480000, columns=21
memory_mb=178.71


## Nulls

In [3]:
null_rates = df.isna().mean().sort_values(ascending=False)
null_rates[null_rates > 0]


lyricist              0.430485
gender                0.401510
bd                    0.399704
composer              0.227258
isrc                  0.078419
source_screen_name    0.055650
genre_ids             0.016271
source_system_tab     0.003344
source_type           0.002954
name                  0.000177
language              0.000040
song_length           0.000035
artist_name           0.000035
dtype: float64

## Target balance

In [4]:
balance = df["target"].value_counts(normalize=True)
print(balance)
fig, ax = plt.subplots(figsize=(4, 3))
balance.sort_index().plot(kind="bar", ax=ax, color=["#4C78A8", "#F58518"])
ax.set_xlabel("target")
ax.set_ylabel("share")
ax.set_title("Repeat-listen target balance")
plt.tight_layout()
plt.show()


target
1    0.503517
0    0.496483
Name: proportion, dtype: float64


/var/folders/sz/q9zb9htj2ns037y0vgdhdlbm0000gn/T/ipykernel_28174/290062047.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Numeric distributions

In [5]:
numeric_cols = ["bd", "song_length", "registration_init_time", "expiration_date"]
df[numeric_cols].describe()


,bd,song_length,registration_init_time,expiration_date
count,288142.000000,479983.000000,4.800000e+05,4.800000e+05
mean,28.718077,245.034799,2.012811e+07,2.017158e+07
std,8.626286,66.482057,3.010705e+04,3.837481e+03
min,2.000000,1.393000,2.004033e+07,2.004102e+07
25%,23.000000,214.595000,2.011070e+07,2.017091e+07
50%,27.000000,241.684000,2.013102e+07,2.017093e+07
75%,33.000000,271.986000,2.015102e+07,2.017101e+07
max,95.000000,7575.835000,2.017013e+07,2.020102e+07


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
df["bd"].clip(-50, 120).hist(bins=40, ax=axes[0])
axes[0].set_title("bd (age), clipped to [-50, 120]")
df["song_length"].clip(0, 600).hist(bins=40, ax=axes[1])
axes[1].set_title("song_length (s), clipped to 600s")
plt.tight_layout()
plt.show()


/var/folders/sz/q9zb9htj2ns037y0vgdhdlbm0000gn/T/ipykernel_28174/2032692105.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


`bd` has a long tail of implausible values (<=0 or >100) - see `modeling/01-data.md` for the exact share. `song_length` is right-skewed as expected for a mixed-duration music catalog.

## Categorical distributions

In [7]:
categorical_cols = ["source_system_tab", "source_screen_name", "source_type",
                     "city", "gender", "registered_via", "language"]
for c in categorical_cols:
    print(f"--- {c} (cardinality={df[c].nunique()}) ---")
    print(df[c].value_counts(dropna=True).head(5))
    print()


--- source_system_tab (cardinality=8) ---
source_system_tab
my library     239942
discover       141451
search          40719
radio           30863
listen with     13940
Name: count, dtype: int64

--- source_screen_name (cardinality=19) ---
source_screen_name
Local playlist more     210243
Online playlist more     84406
Radio                    30701
Album more               27386
Search                   19560
Name: count, dtype: int64

--- source_type (cardinality=12) ---
source_type
local-library      147296
online-playlist    127994
local-playlist      70280
radio               31279
album               31095
Name: count, dtype: int64

--- city (cardinality=21) ---
city
1     170519
13     74807
5      53800
4      35337
15     31382
Name: count, dtype: int64

--- gender (cardinality=2) ---
gender
male      149857
female    137418
Name: count, dtype: int64

--- registered_via (cardinality=5) ---
registered_via
9     183270
7     166087
3      81188
4      48483
13       972
Name: c

In [8]:
print(f"genre_ids cardinality={df['genre_ids'].nunique()}")
print(f"artist_name cardinality={df['artist_name'].nunique()}")
df["artist_name"].value_counts().head(10)


genre_ids cardinality=370
artist_name cardinality=15239


artist_name
Various Artists     20029
周杰倫 (Jay Chou)      12197
五月天 (Mayday)        11986
林俊傑 (JJ Lin)         7528
田馥甄 (Hebe)           6779
aMEI (張惠妹)           5483
陳奕迅 (Eason Chan)     4887
玖壹壹                  4622
G.E.M.鄧紫棋            4305
BIGBANG              4021
Name: count, dtype: int64

## Quality flags

In [9]:
bd_bad = ((df["bd"] <= 0) | (df["bd"] > 100)).mean()
print(f"implausible bd rate: {bd_bad:.1%}")
dupe = df.duplicated(subset=["msno", "song_id"]).sum()
print(f"duplicate (msno, song_id) rows: {dupe}")


implausible bd rate: 0.0%


duplicate (msno, song_id) rows: 0
